# Procesamiento de datos - Capa Silver

Este notebook lee los Parquet de la capa Bronze, normaliza los indicadores del Banco Mundial y consolida una tabla única en Delta Lake para la capa Silver.

In [8]:
from functools import reduce
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

from delta import configure_spark_with_delta_pip

active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()

builder = SparkSession.builder \
    .appName("silver-worldbank-processing") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"
SILVER_OUTPUT_PATH = SILVER_ROOT / "economic_indicators"

SILVER_ROOT.mkdir(parents=True, exist_ok=True)

INDICATOR_COLUMN_MAP = {
    "pib_per_capita": "gdp",
    "indice_gini": "gini",
    "desempleo": "unemployment",
    "inflacion": "inflation",
    "exportaciones": "exports",
    "importaciones": "imports",
    "poblacion": "population",
}

AGGREGATE_CODES = {
    "ARB", "CEB", "CSS", "EAP", "EAR", "EAS", "ECA", "ECS", "EMU", "EUU",
    "FCS", "HIC", "IBD", "IBT", "IDA", "IDB", "IDX", "LAC", "LCN", "LDC",
    "LIC", "LMY", "LTE", "LMC", "MEA", "MIC", "MNA", "NAC", "OED", "OSS",
    "PST", "SAS", "SSA", "SSF", "UMC", "WLD", "AFE", "AFR", "ANZ", "CHI"
}

AGGREGATE_NAMES = {
    "world",
    "high income",
    "upper middle income",
    "lower middle income",
    "low income",
    "latin america & caribbean",
    "middle east & north africa",
    "europe & central asia",
    "east asia & pacific",
    "south asia",
    "sub-saharan africa",
    "north america",
    "european union",
}


def standardize_world_bank_indicator(df, metric_name):
    country_name_col = next((col for col in df.columns if col.lower() in {"country name", "country_name"}), None)
    country_code_col = next((col for col in df.columns if col.lower() in {"country code", "country_code"}), None)

    if country_name_col is None or country_code_col is None:
        raise ValueError(f"No se encontraron columnas de país en el dataset {metric_name}: {df.columns}")

    year_columns = sorted([col for col in df.columns if col.isdigit() and len(col) == 4])
    if not year_columns:
        raise ValueError(f"No se encontraron columnas de año en el dataset {metric_name}: {df.columns}")

    metric_df = df

    for column_name in ["Indicator Name", "Indicator Code"]:
        if column_name in metric_df.columns:
            metric_df = metric_df.drop(column_name)

    metric_df = metric_df.select(country_name_col, country_code_col, *year_columns)

    year_structs = [
        F.struct(
            F.lit(int(year)).alias("year"),
            F.col(f"`{year}`").alias("value"),
        )
        for year in year_columns
    ]

    metric_df = metric_df.select(
        F.trim(F.col(country_name_col)).alias("country_name"),
        F.upper(F.trim(F.col(country_code_col))).alias("country_code"),
        F.explode(F.array(*year_structs)).alias("year_value"),
    )

    metric_df = (
        metric_df
        .select(
            "country_name",
            "country_code",
            F.col("year_value.year").cast("int").alias("year"),
            F.col("year_value.value").cast("double").alias("value"),
        )
        .filter(F.col("value").isNotNull())
        .filter(F.col("country_code").isNotNull())
        .filter(F.length(F.col("country_code")) == 3)
        .filter(~F.col("country_code").isin(*AGGREGATE_CODES))
        .filter(~F.lower(F.col("country_name")).isin(*AGGREGATE_NAMES))
        .dropDuplicates(["country_code", "year"])
        .withColumnRenamed("value", metric_name)
    )

    return metric_df


print(f"Project root: {PROJECT_ROOT}")
print(f"Bronze root: {BRONZE_ROOT}")
print(f"Silver output path: {SILVER_OUTPUT_PATH}")

Project root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation
Bronze root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze
Silver output path: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\silver\economic_indicators


In [9]:
indicator_dfs = []

for bronze_name, metric_name in INDICATOR_COLUMN_MAP.items():
    bronze_path = BRONZE_ROOT / bronze_name
    if not bronze_path.exists():
        raise FileNotFoundError(f"No existe la carpeta Bronze para {bronze_name}: {bronze_path}")

    metric_source = spark.read.parquet(str(bronze_path))
    metric_df = standardize_world_bank_indicator(metric_source, metric_name)
    indicator_dfs.append((metric_name, metric_df))

    print(f"[OK] {bronze_name} -> {metric_name} | filas: {metric_df.count()}")

print("Procesamiento individual de indicadores completado.")

[OK] pib_per_capita -> gdp | filas: 12146
[OK] indice_gini -> gini | filas: 2430
[OK] desempleo -> unemployment | filas: 6846
[OK] inflacion -> inflation | filas: 11737
[OK] exportaciones -> exports | filas: 9294
[OK] importaciones -> imports | filas: 9278
[OK] poblacion -> population | filas: 9278
Procesamiento individual de indicadores completado.


In [10]:
metric_dfs = dict(indicator_dfs)
base_metric = "gdp"

if base_metric not in metric_dfs:
    raise KeyError("El dataset base 'gdp' no fue generado correctamente.")

join_keys = ["country_code", "country_name", "year"]
silver_consolidated = metric_dfs[base_metric]

for metric_name, metric_df in indicator_dfs:
    if metric_name == base_metric:
        continue
    silver_consolidated = silver_consolidated.join(metric_df, on=join_keys, how="left")

silver_consolidated = (
    silver_consolidated
    .select(
        "country_code",
        "country_name",
        "year",
        "gdp",
        "gini",
        "unemployment",
        "inflation",
        "exports",
        "imports",
        "population",
    )
    .dropDuplicates(["country_code", "year"])
    .orderBy("country_code", "year")
)

In [11]:
(
    silver_consolidated.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(str(SILVER_OUTPUT_PATH))
)

print(f"[OK] Silver consolidado guardado en Delta Lake: {SILVER_OUTPUT_PATH}")

[OK] Silver consolidado guardado en Delta Lake: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\silver\economic_indicators
